# Go deeper · Make the guard all yours

In the session notebooks (1.1 to 1.4), you wrote two pieces of the guard, the hook (Cell 1.1) and
OFAC's 50% rule (Cell 2.3). Three more pieces ran from **reference answers**, and the session said so
each time it used one (*"Ran the reference ...; you write it in Go deeper ..."*). These are the pieces
that decide what the guard checks, what it does when the check can't run, and what evidence it gives
for a block.

**The goal of this notebook** is that by the end, the guard that deploys runs only code you wrote.
Cell G.0 shows where things stand now, and Cell G.9 shows where they ended up. After that,
`3 - Run it from Kiro (optional).ipynb` deploys exactly those files as an MCP server for another
agent. Along the way, we'll also write one rule as a W3C SHACL shape, to see what a standard
constraint language can and can't express, and land OFAC's real SDN list in S3, to see what a
production pipeline can download and what it has to get elsewhere. The notebook takes about 30 minutes.

| Part | Cell | You write | In the session it ran in |
|---|---|---|---|
| 1 · The hook's decisions | G.1 | `should_check`: which tool calls your hook checks | every send (Cells 1.3, 2.6, 3.4) |
| | G.2 | `run_report`: the report call, failing closed | 4.3 · Neptune is down |
| 2 · The evidence | G.3 | the blocked chain: the sanctions evidence, in openCypher | 2.6 · Teach it the rule |
| 3 · The same rule, as a standard | G.4 | the UFLPA rule as a SHACL shape | 3.1 · One rule, as a query |
| 4 · When the business changes | G.5, G.6 | nothing: a warehouse sells out; the same supplier delivers to another site | 4.4 · Deploy and resend |
| 5 · The real sanctions list | G.7, G.8 | nothing: land today's SDN list from OFAC in S3, and see what it does and doesn't say | 2 · Computing OFAC's 50 percent rule |

**How each exercise works.** Each cell gives you a scaffold and marks the parts you write with ✏️.
The check runs your answer beside the reference on the same data, and prints PASS or FAIL for each
case. When everything passes, your answer replaces the reference in the file the guard runs. If a
case fails, your version is kept as `.mine`, the reference stays in place, and a solution you can
reveal sits under the cell. Nothing done here can break the session notebooks.

**Cell G.0 · Connect, and see where you are.** This is the same Connect cell as in the session, with
every rule installed. It's followed by a table of every file the deployed guard runs, showing whose
each file is right now.

*Predict: after the session, which files should say "yours"?*

In [ ]:
# Cell G.0 · Connect, and see where you are
import sys, os; sys.path.insert(0, os.path.join(os.path.expanduser("~"), "SageMaker", "workshop", "helpers"))
from session import connect
connect(globals(), all_rules=True)

show_whose("The files your guard runs, before Go deeper")

> **What you should see.** `neptune_guard.py` and `queries/ownership_50pct.cypher` say *yours* if you wrote them in the
> session, and `guard_rules.py` and `queries/violations.cypher` say *the reference*. `recommend.cypher`,
> the fulfilment ranking, is given.
>
> **Why it matters.** The two reference files hold the three pieces you write in this notebook. Each file's fingerprint,
> the first 12 characters of its SHA-256, changes when your answer goes in, so the change is easy to
> see.

## Part 1 · The hook's decisions

Before the hook decides anything, it asks `guard_rules.py` two questions, namely *does this call need
a check?* and *what does the report say?* In the session, both answers came from the reference. They
look small, but each one is a place where a guard can go wrong in production.

**Cell G.1 · ✏️ Which calls to check.** `should_check(tool, args)` decides which tool calls the hook
gates at all. If it errs in one direction, a consequential action slips past the guard. If it errs in
the other, the guard blocks harmless calls until people work around it.

The agent has two action tools, and the rule for each is different.

- `draft_purchase_order` always commits AnyCompany to a supplier, so every call is checked.
- `send_offer_reply` commits AnyCompany only when it **accepts** an offer. The call's `decision`
  argument says whether it accepts or declines, and declining commits to nothing, even with a blocked
  supplier.
- Anything else, such as `recommend_alternatives`, only reads, so it isn't checked.

**Write** `should_check` so that it returns `True` or `False`. When you run the cell, `save_rule`
puts the function in `guard_rules.py`, and says in red if the starter hasn't been filled in yet. Then
`check_guard()` builds the agent with your hook and calls the tools directly, with no model, on seven
cases.

*Predict: the starter returns `False` for everything. Which of the seven cases will fail, and does
that make the guard too strict or too loose?*

In [ ]:
# Cell G.1 · ✏️ Which calls to check
def should_check(tool, args):
    return False   # ✏️ True only for the calls that commit AnyCompany to a supplier

save_rule(should_check)
check_guard()

<details>
<summary>🔑 Reveal the Cell G.1 solution</summary>



```python
def should_check(tool, args):
    return (tool == "draft_purchase_order"
            or (tool == "send_offer_reply" and args.get("decision") == "accept"))
```

</details>

> **What you should see.** The cell shows seven PASS rows. The four-violation order, accepting AnyCompany Hardware's offer, the
> unknown supplier and the Neptune-down case are all blocked, while the clean PO, the warning-only PO and
> the declined offer run. The text the model reads for the blocked order appears below the table.
>
> **Why it matters.** With the starter, every blocked case fails, because a guard that checks nothing is too loose, and
> a demo wouldn't reveal it. Gating on the **arguments**, and not just the tool name, is what lets the
> guard block the acceptance of a bad offer without blocking a polite decline.

**Cell G.2 · ✏️ The report call, failing closed.** `run_report(args)` turns the tool call's arguments
into the validation report. The hook already blocks when this function raises (Cell 1.1), so the
question here is **when it should raise**.

These are the pieces you'll need, and the questions the function has to answer.

- `validation_report(supplier_name, product_name, region_name, quantity)` produces the report, in
  the same way Cell 3.3 called it, and it's already imported in `guard_rules.py`.
- `args` holds the tool call's arguments: `supplier_name`, `product_name`, `region_name`, and for a
  PO, `quantity`.
- **If the model leaves out the supplier, product or region,** what should happen? An order the guard
  can't fully identify shouldn't go through, and quietly checking the wrong thing is worse than
  blocking. Think about how the way you read each argument decides which of those happens.
- **Offer replies have no quantity,** and the stock rule only runs when there is one, so the function
  needs a sensible value when `quantity` is missing.

*Predict: if `run_report` returned `[]` whatever happened, which cases in the check would fail?*

In [ ]:
# Cell G.2 · ✏️ The report call, failing closed
def run_report(args):
    return []   # ✏️ the report for this call's order; a missing supplier, product or region must raise

save_rule(run_report)
check_guard()

<details>
<summary>🔑 Reveal the Cell G.2 solution</summary>

<code>validation_report</code> is imported at the top of <code>guard_rules.py</code>.

```python
def run_report(args):
    return validation_report(args["supplier_name"], args["product_name"],
                             args["region_name"], args.get("quantity", 0))
```

</details>

> **What you should see.** The cell shows seven PASS rows again, this time with your `run_report`. The Neptune-down case is
> blocked by the `except` in your hook, because the report raised.
>
> **Why it matters.** An empty report means "nothing is wrong", so a `run_report` that swallows errors and returns `[]`
> fails **open**. Every order passes, and the guard still looks healthy. Failing closed needs both
> halves, with the hook blocking on an exception and `run_report` raising when it can't do its job.

## Part 2 · The evidence

When the guard blocks an order for sanctions, the block message carries **evidence**, which is the
chain of owners from the supplier up to the listed company. The evidence lets the model explain the
block, and it's what a compliance reviewer checks. In the session, it came from a reference subquery
inside `violations.cypher`, and in this part we write it ourselves.

The difficulty is that a supplier can have several ownership paths up to a sanctioned company, and
not all of them are honest evidence. We'll look at them first.

**Cell G.3a · The candidate paths.** This cell finds every path from two suppliers up `OWNED_BY`, at
any depth, to a company on the sanctions list. For each path, the query returns the names along it,
whether the 50% rule blocked each one, and the stake on each edge. `show_candidate_paths` draws one
row per path, with every blocked entity in red. AnyCompany Components is blocked, and AnyCompany
Electronics isn't.

*Predict: which of these paths would be honest evidence for a block?*

In [ ]:
# Cell G.3a · The candidate paths
CANDIDATES = """
MATCH chain = (s:Supplier {name: $name})-[:OWNED_BY*1..]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
RETURN [n IN nodes(chain) | n.name] AS names,
       [n IN nodes(chain) | coalesce(n.blocked, false)] AS blocked,
       [r IN relationships(chain) | r.share_pct] AS stakes
ORDER BY length(chain)"""

for name in ["AnyCompany Components", "AnyCompany Electronics"]:
    show_candidate_paths(name, oc(CANDIDATES, name=name))

> **What you should see.** AnyCompany Components has two paths, and every box on both of them is red, so each is tagged *all
> blocked*. AnyCompany Electronics has a path too, but it and AnyCompany Electronics Group are grey,
> because the sanctioned firm holds only 20%, so its path is tagged *a gap*.
>
> **Why it matters.** A path to a listed company isn't evidence of a block, which was option B's mistake in Cell 2.2.
> Honest evidence is a path where the supplier **and every owner after it** is blocked, because that's
> the chain the 50% rule actually followed. That filter is what we write next.

**Cell G.3 · ✏️ The blocked chain.** The report runs this subquery once for each supplier `s`. The
walk is given, and it's the same pattern as in Cell G.3a. **Write** the `WHERE` and the `ORDER BY`.

1. Keep a chain only when the supplier `s` itself is blocked, because a supplier that isn't blocked
   needs no evidence.
2. Keep a chain only when every entity after `s` on it is blocked. This Neptune engine has no `all()`
   list predicate, so count the blocked entities instead and compare the count with how many there
   should be. `nodes(chain)`, `tail()`, `length()` and a list comprehension with a `WHERE` are the
   pieces, and section 4 of the cheat sheet, "Every element matches", shows the pattern. Remember that
   the last node on the chain is the `SanctionsList`, which is never `blocked`.
3. Order the chains so the nearest one comes first. The `LIMIT 1` is given.

The check runs your subquery and the reference's on six suppliers (the two named ones, AnyCompany
Supply Co, and three blocked suppliers from the registry) and compares the chains. When they all
match, your subquery is spliced into `queries/violations.cypher` in place of the reference.

In [ ]:
# Cell G.3 · ✏️ The blocked chain
BLOCKED_CHAIN = """
OPTIONAL MATCH chain = (s)-[:OWNED_BY*1..]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
// ✏️ WHERE: s is blocked, and every entity after s on the chain is blocked
RETURN chain
// ✏️ ORDER BY: the nearest chain first
LIMIT 1
"""
check_blocked_chain(BLOCKED_CHAIN)

<details>
<summary>🔑 Reveal the Cell G.3 solution</summary>

The value of <code>BLOCKED_CHAIN</code>.

```cypher
OPTIONAL MATCH chain = (s)-[:OWNED_BY*1..]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
WHERE s.blocked = true
  AND size([n IN tail(nodes(chain)) WHERE n.blocked = true]) = length(chain) - 1
RETURN chain
ORDER BY length(chain)
LIMIT 1
```

</details>

> **What you should see.** The cell shows six PASS rows, with a chain of blocked entities for each blocked supplier and *none*
> for AnyCompany Electronics and AnyCompany Supply Co. It then reports *Put into
> queries/violations.cypher, in place of the reference*, and the ten-order report check passes.
>
> **Why it matters.** The evidence query is now yours, so every sanctions block the guard makes, in the notebook, the
> deployed agent and the MCP server, explains itself with the chain your query picked. Without the
> `WHERE`, the evidence for a block could be a path through an owner that OFAC doesn't block.

## Part 3 · The same rule, as a standard

The report is the property-graph counterpart of **SHACL**, the W3C Shapes Constraint Language, in
which each result has a focus, a rule, a severity and a message. Many organisations keep their
constraints in SHACL, so in this part we'll find out whether a rule like UFLPA can be written there.

The check exports each order's supply chain from Neptune as RDF, as an `nc:PurchaseOrder` with its
supplier and region, plus every `nc:sourcesFrom` edge. It validates the data with `pyshacl`, and
compares the verdicts with the report's UFLPA row.

**Cell G.4 · ✏️ The UFLPA rule as a SHACL shape.** The shape is a SHACL-SPARQL constraint, in which
the `SELECT` returns one row for each violation, like the openCypher rule in Cell 3.1. The parts
around it are given, namely the target (every `nc:PurchaseOrder`), the U.S.-region condition and the
message.

**Write** the triple patterns that make the walk. Together, they need to say three things.

1. The supplier sources from a maker, `?value`, at **any** tier. In SPARQL, a `+` after a predicate
   makes it a **property path**, meaning one or more hops, the counterpart of `*1..` in openCypher.
2. That maker is on a list, and its name is bound to `?name`, which the message uses.
3. The list is the one named `"UFLPA Entity List"`.

The vocabulary in the exported data is `nc:sourcesFrom` (supplier or company to the company it
sources from), `nc:onList` (company to list) and `nc:name` (on both companies and lists). The
variables `?supplier`, `?value` and `?name` are already used by the rest of the shape.

Run the cell to save `queries/uflpa_shape.ttl`.

In [ ]:
%%writefile queries/uflpa_shape.ttl
# Cell G.4 · ✏️ The UFLPA rule as a SHACL shape
# The UFLPA rule as a SHACL-SPARQL shape: an order to a U.S. region is a violation when its
# supplier sources, at any tier, from a company on the UFLPA Entity List. The property path
# nc:sourcesFrom+ walks the chain to any depth, like -[:SOURCES_FROM*1..]-> in openCypher.
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix nc: <https://anycompany.example/procurement#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

nc: a owl:Ontology ;
    sh:declare [ sh:prefix "nc" ; sh:namespace "https://anycompany.example/procurement#"^^<http://www.w3.org/2001/XMLSchema#anyURI> ] .

nc:OrderUflpaShape a sh:NodeShape ;
    sh:targetClass nc:PurchaseOrder ;
    sh:sparql [
        a sh:SPARQLConstraint ;
        sh:severity sh:Violation ;
        sh:message "Sources, at some tier, from {?name}, on the UFLPA Entity List" ;
        sh:prefixes nc: ;
        sh:select """
            SELECT $this ?value ?name WHERE {
                $this nc:supplier ?supplier ; nc:region ?region .
                ?region nc:country "US" .
                # ✏️ ?supplier sources from ?value at any tier; ?value is on the UFLPA Entity List;
                #    bind ?value's name to ?name
            }
        """ ;
    ] .

<details>
<summary>🔑 Reveal the Cell G.4 solution</summary>

The three lines that replace the ✏️ comment.

```turtle
?supplier nc:sourcesFrom+ ?value .
                ?value nc:onList ?list ; nc:name ?name .
                ?list nc:name "UFLPA Entity List" .
```

</details>

**Check G.4 · Your shape beside the report.** The check uses four orders, for AnyCompany Components
to US-East, AnyCompany Devices to US-East and to EU-Central, and AnyCompany Supply Co to US-East. For
each one, it shows the shape's verdict beside the report's.

**What runs, and where.**

- `check_uflpa_shape()`, in [`helpers/session.py`](helpers/session.py), uses [`helpers/shacl_check.py`](helpers/shacl_check.py).
  `supply_chain_rdf()` reads each order's supply chain from Neptune and builds it as RDF with
  RDFLib, and `shacl_validate()` runs pyshacl with your shape on it.
- The shape's verdict is compared with the `uflpa` row from `validation_report()` in [`helpers/verify.py`](helpers/verify.py).

*Predict: which of the four orders should the shape flag?*

In [ ]:
# Check G.4 · your shape beside the report
check_uflpa_shape()

> **What you should see.** The shape and the report agree on all four orders. AnyCompany Components and AnyCompany Devices are
> flagged for US-East, both because of AnyCompany Polysilicon Co., and AnyCompany Devices to EU-Central
> and AnyCompany Supply Co conform.
>
> **Why it matters.** A walk of any depth carries over to SHACL directly, through a property path. OFAC's 50% rule doesn't,
> because whether an entity is blocked depends on whether its owners are blocked, summed at every level
> until nothing changes. A SHACL 1.0 shape checks the data as it is, and can't compute that. In Neptune,
> it's a single query run to a fixpoint (Cell 2.3). Rules that are walks can move between standards,
> while rules that have to be computed need the graph.

## Part 4 · When the business changes

Procurement doesn't stand still. Between one request and the next, a warehouse ships out its stock,
and the same supplier is asked to deliver to a different site. Neither event should mean changing the
guard's code or redeploying it. The guard should simply give the right answer the next time it's
asked.

The two cells in this part play out both events, and there's nothing to write. In each, the business
changes, and we ask the same guard again.

**Cell G.5 · A warehouse sells out.** In the session, when the deployed agent blocked the order (Cell
4.4), it suggested AnyCompany Supply Co instead, a compliant supplier with 800 units in its Virginia
warehouse, a day from Ashburn. That suggestion comes from a ranking. `recommend.cypher` finds
compliant suppliers and the warehouses that carry the product, nearest first, and the DynamoDB stock
table says which of those warehouses have the units.

Now another team's order empties the Virginia warehouse. The cell takes four steps.

1. It ranks the alternatives for the order, as the agent did.
2. It records the shipment the way the warehouse system would, with one DynamoDB `UpdateItem` that
   sets Virginia's count to 0.
3. It ranks the alternatives again, with the same query and the same code.
4. It puts the 800 units back, so the rest of the notebook sees the original stock.

**What runs, and where.**

- `move_stock(...)`, in [`helpers/session.py`](helpers/session.py), ranks the alternatives with
  `recommend_alternatives()` in [`helpers/verify.py`](helpers/verify.py). That runs
  [`your-guard/queries/recommend.cypher`](your-guard/queries/recommend.cypher) for compliant suppliers and their warehouses,
  nearest first, and reads the counts from DynamoDB in batches until it has enough suppliers.
- It then sets the Virginia count with one DynamoDB `UpdateItem`, ranks again, and puts the count
  back in a `finally` block.

*Predict: AnyCompany Supply Co is still compliant, and still has stock in Frankfurt. Should the agent
still suggest it for a build-out in Ashburn?*

In [ ]:
# Cell G.5 · A warehouse sells out
VIRGINIA = {"warehouse_id": "wh-ac-supply-iad", "product_id": "product-server-hardware"}   # the graph's ids
move_stock(VIRGINIA, units=0)

> **What you should see.** AnyCompany Supply Co tops the first ranking and is missing from the second. The two drawings show
> why. Before the shipment, its Virginia warehouse has 800 units, one day from US-East along
> `LOCATED_IN` and `SHIPS_TO`. After it, Virginia is red with 0 units, and the nearest stock is in
> Frankfurt, five days away, so faster suppliers move ahead. The graph's paths are the same in both
> drawings, and only the count changed. The last line confirms that the 800 units are back.
>
> **Why it matters.** An agent that suggested AnyCompany Supply Co anyway would send procurement to a supplier that can't
> deliver this week. The graph knows which suppliers are allowed and where they ship from, and the stock
> table knows what's on the shelf right now. The guard reads both when it acts, so a shipment recorded a
> second ago changes the next suggestion.

**Cell G.6 · The same supplier, another site.** AnyCompany's build-outs aren't all in Ashburn. Run
the report for AnyCompany Devices supplying Server Hardware to the Ashburn site (US-East), then change
only `region` to `"EU-Central"`, for a build-out in Frankfurt, and run it again. After that, try orders
of your own, such as any `Supplier 0xxxxx` from the registry, or a quantity larger than a supplier's
stock.

**What runs, and where.**

- `full_report(...)`, in [`helpers/session.py`](helpers/session.py), calls `validation_report()` in
  [`helpers/verify.py`](helpers/verify.py), the same report as in Cell 3.3, and shows it the same way.

*Predict: the supplier, the product and the quantity stay the same. Can the answer be different?*

In [ ]:
# Cell G.6 · The same supplier, another site
MY_ORDER = dict(supplier="AnyCompany Devices", product="Server Hardware", region="US-East", quantity=10)
full_report(MY_ORDER)

> **What you should see.** For Ashburn (US-East), the report shows one `uflpa` violation, and the drawing shows its evidence,
> the supply-chain path through AnyCompany Modules and AnyCompany Materials to the listed maker. For
> Frankfurt (EU-Central), no rule is broken, so there's nothing to draw.
>
> **Why it matters.** The UFLPA bars goods from entering the United States, so the same supplier is a no for Ashburn and a
> yes for Frankfurt. A single "approved" or "blocked" flag on the supplier can't express that. A guard
> that checks the supplier, the product, the site and the quantity together can, and it explains why
> each time.

## Part 5 · The real sanctions list

Every company in this registry is synthetic, and the FAQ explains why. In this part, we take the
first step of the production pattern with real data. OFAC publishes the SDN list as files on its
[Sanctions List Service](https://sanctionslist.ofac.treas.gov/), and updates them whenever it
designates or removes someone. No regulator offers an API that answers "is this supplier blocked?",
so a production pipeline downloads the list on a schedule and lands each download in Amazon S3, as a
dated snapshot it can audit and replay.

We stop there, on purpose. The guard can only use a listed entity once it's linked to a company in
your registry, which takes ownership data and entity resolution, the work of matching names and
identifiers across sources. This registry is synthetic, so there's nothing real to link the list to.

**Cell G.7 · Land today's SDN list in S3.** The cell downloads `SDN.CSV` and writes it, unchanged, to
the workshop's data bucket under a key with today's date. It records where the file came from and its
SHA-256 checksum as object metadata, so the snapshot can be checked later. It then lists the
snapshots under the prefix. In your system, a scheduled job, such as an Amazon EventBridge rule
that runs an AWS Lambda function, does the same.

*Predict: the cell runs again tomorrow. What happens to today's snapshot?*

In [ ]:
# Cell G.7 · Land today's SDN list in S3
import datetime, hashlib, urllib.request, boto3
SDN_URL = "https://sanctionslistservice.ofac.treas.gov/api/PublicationPreview/exports/SDN.CSV"
raw = urllib.request.urlopen(SDN_URL, timeout=60).read()

s3 = boto3.client("s3", region_name=os.environ["AWS_REGION"])
bucket, prefix = os.environ["DATA_BUCKET"], "guardrail/lists/ofac-sdn/"
key = f"{prefix}{datetime.date.today().isoformat()}/SDN.CSV"
s3.put_object(Bucket=bucket, Key=key, Body=raw, ContentType="text/csv",
              Metadata={"source-url": SDN_URL, "sha256": hashlib.sha256(raw).hexdigest()})

for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefix).get("Contents", []):
    print(f"s3://{bucket}/{obj['Key']}  {obj['Size']:,} bytes")

> **What you should see.** One line per snapshot, with today's at the end, at about 5.8 MB. Running the cell again on the same
> day overwrites today's snapshot, and running it tomorrow adds a new one beside it, so the history
> stays.
>
> **Why it matters.** The snapshot is the system of record for what the list said on a given day. When the guard blocks or
> passes an order, you can show which version of the list it was judged against, and you can replay a
> decision against an older snapshot. The rule's result in Neptune is derived from these files, and S3
> is where they live.

**Cell G.8 · What the snapshot holds, and what it doesn't.** The cell reads the snapshot back from
S3, as a downstream job would, and counts its entries by type. It then counts two things in the
remarks: entries linked to another entry (`Linked To:`) and entries that give a percentage.

*Predict: the 50% rule needs to know how much of each company its owners hold. How many entries say?*

In [ ]:
# Cell G.8 · What the snapshot holds, and what it doesn't
import csv, io, collections
body = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8", "replace")
rows = [r for r in csv.reader(io.StringIO(body)) if len(r) == 12]   # OFAC writes "-0-" for empty

types = collections.Counter("entity" if r[2].strip() == "-0-" else r[2].strip() for r in rows)
linked = sum("Linked To:" in r[11] for r in rows)
with_pct = sum("%" in r[11] for r in rows)
show_results([(t, f"{n:,}") for t, n in types.most_common()]
             + [("linked to another entry", f"{linked:,}"), ("giving an ownership percentage", f"{with_pct:,}")],
             ["in today's snapshot", "entries"], title=key)

> **What you should see.** About 19,500 entries, of which about 10,000 are entities, 7,500 individuals, 1,600 vessels and 340
> aircraft. About 8,700 are linked to another entry, and none gives a percentage.
>
> **Why it matters.** The list says who is blocked, and sometimes who they're linked to, but never how much of a company
> anyone owns, and a company blocked by ownership alone never appears on it, which is why the 50% rule
> exists. To use a snapshot, a production pipeline matches its entries to the companies in its own
> ownership data, which teams license from commercial providers, writes the matches into the graph as
> `SANCTIONED_BY` edges, and reruns the rule from Step 2. The registry's `SANCTIONED_BY` edges stand in
> for that result.

## Where you got to

**Cell G.9 · What's yours now.** This cell shows the table from Cell G.0 again.

**What runs, and where.**

- `show_whose(...)`, in [`helpers/session.py`](helpers/session.py), lists every file the deployed guard runs, says
  whether each one is yours or the reference, and shows its fingerprint, the first 12 characters of
  its SHA-256.

In [ ]:
# Cell G.9 · What's yours now
show_whose("The files your guard runs, after Go deeper")

> **What you should see.** `neptune_guard.py`, `queries/ownership_50pct.cypher`, `guard_rules.py` and `queries/violations.cypher`
> all say *yours*, with new fingerprints. Any file that still says *the reference* points to an exercise
> worth going back to.
>
> **Why it matters.** Every decision the guard makes now comes from code you wrote, including which calls it checks, how
> it fails, which rule it computes and what evidence it gives. Next, open
> `3 - Run it from Kiro (optional).ipynb`, where Cell K.2 deploys exactly these files as an MCP server
> and Kiro, another AI agent, is held to them.